# Week 32, Day 3: Compartmental models: discretising and simulating a neuron

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Explicit and implicit Euler on a fine cable

For a 2 μm dendrite of 100 compartments of 8 μm, compute the explicit stability limit `dt_max`. Run 2 ms of a 0.1 nA step injected into the first compartment with explicit Euler at dt = 25 μs (`V_explicit`), explicit Euler at dt = dt_max / 2 (`V_fine`), and implicit Euler at 25 μs (`V_implicit`, using np.linalg.solve each step).

In [ ]:
import numpy as np
import math
import time
rm, ri, cm = 20000.0, 150.0, 1e-6
d, dx, N = 2e-4, 8e-4, 100
ga = np.pi * d**2 / (4 * ri * dx); gm = np.pi * d * dx / rm; cmp = np.pi * d * dx * cm
G = np.diag(np.full(N, gm + 2 * ga)) - ga * np.eye(N, k=1) - ga * np.eye(N, k=-1)
G[0, 0] -= ga; G[-1, -1] -= ga
I = np.zeros(N); I[0] = 1e-10
dt_max = ...
print(f"explicit Euler is stable only for dt below {dt_max * 1e6:.2f} us")

def explicit(dt, T=2e-3):
    V = np.zeros(N)
    with np.errstate(all="ignore"):
        for _ in range(int(round(T / dt))):
            V = V + dt / cmp * (I - G @ V)
    return V

def implicit(dt, T=2e-3):
    V = np.zeros(N); M = cmp / dt * np.eye(N) + G
    for _ in range(int(round(T / dt))):
        V = ...
    return V

V_explicit = explicit(25e-6)
V_fine = explicit(dt_max / 2)
V_implicit = implicit(25e-6)
print(f"V at the injection site after 2 ms: explicit at 25 us {V_explicit[0]:.3g} V, explicit at dt_max/2 {V_fine[0] * 1e3:.3f} mV, implicit at 25 us {V_implicit[0] * 1e3:.3f} mV")

In [ ]:
import numpy as np
def test_limit():
    "dt_max is the compartment capacitance over the membrane plus twice the axial conductance, and is under 2 microseconds"
    assert abs(dt_max - cmp / (gm + 2 * ga)) < 1e-15 and dt_max < 2e-6

def test_outcome():
    "explicit Euler at 25 us blows up while implicit Euler stays bounded and close to the fine explicit run"
    assert not np.isfinite(V_explicit).all() or np.abs(V_explicit).max() > 1e3
    assert np.abs(V_implicit).max() < 0.1 and abs(V_implicit[0] - V_fine[0]) / V_fine[0] < 0.05

test_limit()
test_outcome()
print("All checks passed")

### Exercise: How much faster?

Solve the implicit-Euler system for a cable of 1,500 compartments with the dense solver and with `solve_tri`, time both, and check they agree.

In [ ]:
def solve_tri(lower, diag, upper, rhs):
    n = len(diag); c = np.empty(n); dd = np.empty(n)
    c[0], dd[0] = upper[0] / diag[0], rhs[0] / diag[0]
    for i in range(1, n):
        m = diag[i] - lower[i - 1] * c[i - 1]
        c[i] = upper[i] / m if i < n - 1 else 0.0
        dd[i] = (rhs[i] - lower[i - 1] * dd[i - 1]) / m
    x = np.empty(n); x[-1] = dd[-1]
    for i in range(n - 2, -1, -1):
        x[i] = dd[i] - c[i] * x[i + 1]
    return x

n = 1500; dt = 25e-6
diag = np.full(n, cmp / dt + gm + 2 * ga); diag[0] -= ga; diag[-1] -= ga
off = np.full(n - 1, -ga)
rhs = np.random.default_rng(0).normal(size=n) * 1e-12
M = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)
t0 = time.perf_counter(); x_dense = np.linalg.solve(M, rhs); t_dense = time.perf_counter() - t0
t0 = time.perf_counter(); x_tri = ...; t_tri = time.perf_counter() - t0
print(f"dense solve {t_dense * 1e3:.1f} ms, tridiagonal {t_tri * 1e3:.1f} ms")

In [ ]:
import numpy as np
def test_agree():
    "the tridiagonal solution agrees with the dense one"
    assert np.allclose(x_tri, x_dense, rtol=1e-8)

def test_speed():
    "the tridiagonal solver is faster than the dense one for 1,500 compartments"
    assert t_tri < t_dense

test_agree()
test_speed()
print("All checks passed")

### Exercise: Distance raises the threshold

`run(site, amp)` simulates 12 ms and returns the voltage at the soma and the tip. Find by bisection (10 steps between 0 and 5 nA) the smallest 1 ms pulse that makes the soma cross 0 mV, injected at the soma (site 0) and at the tip (site N), and store them in `th`. Then record the peak voltage at the soma and at the tip during a spike triggered from the soma.

In [ ]:
Rm = 20000.0; As = math.pi * (20e-4) ** 2; Nd = 50; ddx = 500e-4 / Nd; Ad = math.pi * 2e-4 * ddx
Cc = np.r_[1.0 * As * 1e3, np.full(Nd, 1.0 * Ad * 1e3)]                 # nF
gL = np.r_[0.3e-3 * As * 1e6, np.full(Nd, Ad / Rm * 1e6)]              # uS
EL = np.r_[-54.387, np.full(Nd, -65.0)]
gax = math.pi * (2e-4) ** 2 / (4 * 150 * ddx) * 1e6                     # uS between dendritic compartments
g0 = 1 / (4 * 150 * (ddx / 2) / (math.pi * (2e-4) ** 2)) * 1e6         # soma to first dendritic compartment

def rates(V):
    V = V + 1e-7
    return (0.01 * (V + 55) / (1 - math.exp(-(V + 55) / 10)), 0.125 * math.exp(-(V + 65) / 80),
            0.1 * (V + 40) / (1 - math.exp(-(V + 40) / 10)), 4.0 * math.exp(-(V + 65) / 18),
            0.07 * math.exp(-(V + 65) / 20), 1 / (1 + math.exp(-(V + 35) / 10)))

def run(site, amp, T=12.0, dt=0.01):
    n = Nd + 1; V = np.full(n, -65.0); gn, gmm, gh = 0.3177, 0.0529, 0.5961
    g = np.r_[g0, np.full(Nd - 1, gax)]
    off = -g * dt; diag = Cc + gL * dt; diag[:-1] += g * dt; diag[1:] += g * dt
    out = np.empty((int(T / dt), 2))
    for s in range(len(out)):
        an, bn, am, bm, ah, bh = rates(V[0])
        Iion = np.zeros(n); Iion[0] = (120 * gmm**3 * gh * (V[0] - 50) + 36 * gn**4 * (V[0] + 77)) * As * 1e3   # nA
        gn += dt * (an * (1 - gn) - bn * gn); gmm += dt * (am * (1 - gmm) - bm * gmm); gh += dt * (ah * (1 - gh) - bh * gh)
        Iext = np.zeros(n)
        if 1 <= s * dt < 2:
            Iext[site] = amp
        V = solve_tri(off, diag, off, Cc * V + dt * (Iext - Iion + gL * EL))
        out[s] = V[0], V[-1]
    return out

th = {}
for name, site in (("soma", 0), ("tip", Nd)):
    lo, hi = 0.0, 5.0
    for _ in range(10):
        mid = (lo + hi) / 2
        ...
    th[name] = hi
out = run(0, 1.5 * th["soma"])
soma_peak, tip_peak = out[:, 0].max(), out[:, 1].max()
print(f"threshold for a 1 ms pulse: soma {th['soma']:.3f} nA, tip {th['tip']:.3f} nA ({th['tip'] / th['soma']:.1f} times more)")
print(f"spike from the soma: peak {soma_peak:.1f} mV at the soma, {tip_peak:.1f} mV at the tip")

In [ ]:
import numpy as np
def test_thresholds():
    "the tip needs more than 1.5 times the current that the soma needs"
    assert th["tip"] > 1.5 * th["soma"]

def test_bap():
    "when the soma fires, the spike reaches the passive tip smaller than at the soma"
    assert tip_peak < soma_peak and tip_peak > -65

test_thresholds()
test_bap()
print("All checks passed")